### RAG 

In [1]:
from langchain_core.documents import Document

In [2]:
# doc=Document(
#     page_content="Hello World",
#     metadata={
#         "source":"https://www.google.com"
#     }
# )

In [3]:
# doc

In [4]:
# type(doc)

In [5]:
# Text data

# from langchain_community.document_loaders.text import TextLoader

# loader=TextLoader("data/python.txt",encoding="utfa8")

# document=loader.load()

# document

In [6]:
# from langchain_community.document_loaders.pdf import PyPDFLoader

# pdf_loader=PyPDFLoader("data/research1.pdf")

# document=pdf_loader.load()

# document


### Ingestion Pipeline

In [7]:
from langchain_community.document_loaders.pdf import PyPDFLoader
import os

C:\Users\HP\AppData\Local\Temp\ipykernel_13184\3044880069.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.pdf import PyPDFLoader


### Documents

In [8]:
def load_all_pdfs():
    folder_path="data/pdfs"
    num_docs=0
    all_docs=[]

    for filename in os.listdir(folder_path):

        if filename.lower().endswith(".pdf"):

            pdf_path=os.path.join(folder_path,filename)

            loader=PyPDFLoader(pdf_path)

            doc=loader.load()

            all_docs.extend(doc)
            num_docs+=1

    print(f"Total Docs:{num_docs}")
    print(f"Total pages:{len(all_docs)}")
    return all_docs

In [9]:
all_pdf_documents=load_all_pdfs()

Total Docs:2
Total pages:32


### Chunking

In [10]:
# Chunks

from langchain_text_splitters import RecursiveCharacterTextSplitter

In [11]:
def split_docs(documents,chunk_size=500,chunk_overlap=50):

    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    chunked_docs=text_splitter.split_documents(documents)

    return chunked_docs

In [12]:
chunks=split_docs(all_pdf_documents)

In [13]:
len(chunks)

321

In [14]:
# chunks[0]

### Embedding

In [15]:
from sentence_transformers import SentenceTransformer

In [16]:
class EmbeddingManager:
    def __init__(self,model_name="all-MiniLM-L6-v2"):
        self.model_name=model_name
        print("Loading Model:",self.model_name)
        self.model=SentenceTransformer(model_name)
        print("Embedding Dimension",self.model.get_sentence_embedding_dimension())

    def generate_embeddings(self,text):
        embeddings=self.model.encode(text,show_progress_bar=True)
        print("Embeddings Model:",embeddings.shape)
        return embeddings

In [17]:
embedding_manager=EmbeddingManager()

Loading Model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding Dimension 384


C:\Users\HP\AppData\Local\Temp\ipykernel_13184\704684039.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding Dimension",self.model.get_sentence_embedding_dimension())


### Vector Store

In [18]:
import chromadb
import uuid

In [19]:
class VectorStoreManager:
    def __init__(self,collection_name="pdf_documents",persistent_directory="data/vectorstore"):
        self.collection_name=collection_name
        self.persistent_directory=persistent_directory
        self.collection=None
        self.client=None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persistent_directory,exist_ok=True)

        # make client
        self.client=chromadb.PersistentClient(path=self.persistent_directory)

        # make collection
        self.collection=self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={
                "description":"Vector store collection for pdf embeddings in RAG"
            }
        )

        print("Initialized the vector store with collection",self.collection_name)
        print("Docs in collection",self.collection.count())

    def add_documents(self,documents,embeddings):
        if len(documents)!=len(embeddings):
            raise ValueError("No of documents does not match number of embeddings")

        #store => ids,documents,embeddings,metadata

        ids=[]
        all_metadata=[]
        document_content=[]
        embedding_list=[]

        for i,(doc,embedding) in enumerate(zip(documents,embeddings)):

            doc_id=f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata=dict(doc.metadata)
            metadata["index"]=i
            metadata["Lenght of context"]=len(doc.page_content)
            all_metadata.append(metadata)

            document_content.append(doc.page_content)

            embedding_list.append(embedding.tolist())

        self.collection.add(
            ids=ids,
            metadatas=all_metadata,
            documents=document_content,
            embeddings=embedding_list

        )

        print(f"Total Documents added in vector store:{len(document_content)}")
        print("Docs in collection",self.collection.count())

In [20]:
vector_store=VectorStoreManager()

Initialized the vector store with collection pdf_documents
Docs in collection 321


In [21]:
vector_store

### Data => Documents => chunks => embeddings => Store in vector database

In [22]:
texts=[doc.page_content for doc in chunks]

In [23]:
embeddings=embedding_manager.generate_embeddings(texts)

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Embeddings Model: (321, 384)


In [24]:

vector_store.add_documents(chunks,embeddings)

Total Documents added in vector store:321
Docs in collection 642


### Retrieval Pipeline

In [25]:
from sklearn.metrics.pairwise import cosine_similarity

In [26]:
class RAGRetriever:
    def __init__(self,embedding_manager,vector_store):
        self.embedding_manager=embedding_manager
        self.vector_store=vector_store

    def retrieve(self,query,top_k=5,score_threshold=0.0):

        # query => embeddings

        query_embeddings=self.embedding_manager.generate_embeddings([query])[0]

        # Semantic Search 

        results=self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results=top_k
        )

        # Retrieved Documents

        retrieved_docs=[]

        if results["documents"] and results["documents"][0]:
            doc_ids=results["ids"][0]
            metadatas=results["metadatas"][0]
            documents=results["documents"][0]
            distances=results["distances"][0]

            for i,(doc_id ,metadata ,document ,distance) in enumerate(zip(doc_ids,metadatas,documents,distances)):
                simalarity_score=1-distance

                if simalarity_score>=score_threshold:
                    retrieved_docs.append(
                        {
                            "ids":doc_id,
                            "metadata":metadata,
                            "documents":document,
                            "distance":distance,
                            "simalarity_score":simalarity_score,
                            "ranking":i+1
                        }
                    )

            print(f"Retrieved {len(retrieved_docs)} Documents")

        else:
            print("No documents found")

        return retrieved_docs

In [27]:
rag_retriever=RAGRetriever(embedding_manager,vector_store)

In [28]:
rag_retriever.retrieve("What is RAG")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings Model: (1, 384)
Retrieved 5 Documents


[{'ids': 'doc_002aa8b9-0ee3-4f3a-ab95-0f03ab58ff36',
  'metadata': {'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
   'page': 0,
   'creationdate': '2024-03-28T00:54:45+00:00',
   'creator': 'LaTeX with hyperref',
   'total_pages': 21,
   'Lenght of context': 288,
   'trapped': '/False',
   'keywords': '',
   'index': 88,
   'subject': '',
   'title': '',
   'moddate': '2024-03-28T00:54:45+00:00',
   'author': '',
   'page_label': '1',
   'source': 'data/pdfs\\research2.pdf',
   'producer': 'pdfTeX-1.40.25'},
  'documents': 'and speculate on upcoming trends and innovations.\nOur contributions are as follows:\n• In this survey, we present a thorough and systematic\nreview of the state-of-the-art RAG methods, delineating\nits evolution through paradigms including naive RAG,\narXiv:2312.10997v5  [cs.CL]  27 Mar 2024',
  'distance': 0.4226756691932678,
  'simalarity_score': 0.5773243308067322,
  'ranking': 1},
 {'ids': 'doc_92af

In [29]:
rag_retriever.retrieve("What is Attention?")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings Model: (1, 384)
Retrieved 5 Documents


[{'ids': 'doc_d5ee1a32-2aa0-4f7f-9be0-ce03339e0606',
  'metadata': {'index': 19,
   'producer': 'pdfcpu v0.12.1 dev',
   'language': 'en-US',
   'created': '2017',
   'total_pages': 11,
   'book': 'Advances in Neural Information Processing Systems 30',
   'creator': 'PyPDF',
   'page': 2,
   'published': '2017',
   'page_label': '3',
   'eventtype': 'Poster',
   'publisher': 'Curran Associates, Inc.',
   'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin',
   'subject': 'Neural Information Processing Systems http://nips.cc/',
   'moddate': '2026-07-29T06:50:27+00:00',
   'Lenght of context': 499,
   'source': 'data/pdfs\\research1.pdf',
   'date': '2017',
   'description': 'Paper accepted and presented at the Neural Information Processing Systems Conference (http://nips.cc/)',
   'type': 'Conference Proceedings',
   'creationdate': '2026-07-29T06:50:27+00:00',
   'lastpage': '6008',
   'firstpage': '5998',

### Integrate RAG with LLMs

### OpenAI-GPT

In [33]:
from dotenv import load_dotenv

In [34]:
load_dotenv()

openai_api_key=os.getenv("OPENAI_API_KEY")

In [39]:
# !pip install langchain-openai

In [40]:
from langchain_openai import ChatOpenAI

llm=ChatOpenAI(
    openai_api_key=openai_api_key,
    model="gpt-5.4",
    temperature=0.1,
    max_tokens=1024
)

In [44]:
# generate our retrieval augmented output

def generate_output(query,retriever,llm,top_k=3):
    results=retriever.retrieve(query,top_k)

    context="\n".join([doc["documents"] for doc in results]) if results else ""

    if not context:
        print("Found No relevant context for the query")

    # context+query
    prompt=f"""Use the given context to answer the query 
    context:{context},query:{query}"""

    response=llm.invoke(prompt)

    return response.content

### Groq

In [49]:
# !pip install langchain-groq

In [51]:
groq_api_key=os.getenv("GROQ_API_KEY")

In [63]:
from groq import Groq

client = Groq(api_key=groq_api_key)

models = client.models.list()

for model in models.data:
    print(model.id)

groq/compound
llama-3.3-70b-versatile
llama-3.1-8b-instant
meta-llama/llama-prompt-guard-2-86m
whisper-large-v3-turbo
qwen/qwen3.6-27b
openai/gpt-oss-20b
groq/compound-mini
canopylabs/orpheus-arabic-saudi
openai/gpt-oss-safeguard-20b
meta-llama/llama-prompt-guard-2-22m
whisper-large-v3
openai/gpt-oss-120b
canopylabs/orpheus-v1-english
allam-2-7b


In [64]:
from langchain_groq import ChatGroq

In [70]:
llm=ChatGroq(
    groq_api_key=groq_api_key,
    model="qwen/qwen3.6-27b",
    temperature=0.1,
    max_tokens=1024,
    reasoning_effort=None
)

In [71]:
# generate our retrieval augmented output

def generate_output(query,retriever,llm,top_k=3):
    results=retriever.retrieve(query,top_k)

    context="\n".join([doc["documents"] for doc in results]) if results else ""

    if not context:
        print("Found No relevant context for the query")

    # context+query
    prompt=f"""Use the given context to answer the query 
    context:{context},query:{query}"""

    response=llm.invoke([prompt.format(context=context,query=query)])

    return response.content.replace(".",".\n")

In [72]:
answer=generate_output("What is RAG?",rag_retriever,llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings Model: (1, 384)
Retrieved 3 Documents


In [73]:
answer

'\n<think>\nHere\'s a thinking process:\n\n1.\n  **Analyze User Input:**\n   - **Context:** A snippet from a research paper/survey about RAG (Retrieval-Augmented Generation).\n It mentions:\n     - "thorough and systematic review of the state-of-the-art RAG methods"\n     - "evolution through paradigms including naive RAG, .\n.\n.\n effective RAG framework.\n"\n     - "summarized the current assessment methods of RAG, covering 26 tasks, nearly 50 datasets.\n.\n.\n"\n     - "anticipate future directions for RAG.\n.\n.\n"\n     - "Section II introduces the main concept and current paradigms of RAG.\n"\n   - **Query:** "What is RAG?"\n\n2.\n  **Identify Key Information Needed:**\n   - The user is asking for a definition/explanation of RAG based *only* on the provided context.\n\n   - I need to check if the context explicitly defines RAG.\n\n   - Looking at the context: It mentions "RAG methods", "naive RAG", "effective RAG framework", "assessment methods of RAG", "main concept and current